In [1]:
import os
import json
import re
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

In [2]:
# ====================
# 0. 基础配置
# ====================
model_name = "./model/Mistral-7B/mistralai/Mistral-7B-Instruct-v0___3/"
output_dir = "./output/VLE_LLM/等温等压/随机划分/random_72_aug"

In [3]:
# ====================
# 1. 加载 tokenizer 和模型
# ====================
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Mistral 特殊设置：确保 pad_token 存在且方向正确
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=dtype,
    device_map="auto"
)
model.config.use_cache = False
model.enable_input_require_grads()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [4]:
MAX_LENGTH = 1024
SYSTEM_PROMPT = """You are an expert in thermodynamics and vapor-liquid equilibrium.
Your task is to predict equilibrium properties of binary mixtures from molecular structures and operating conditions.
Return only the requested numerical results.
"""
def build_messages(user_input, assistant_output=None):
    # 将System Prompt直接拼接到User内容
    user_content = SYSTEM_PROMPT.strip() + "\n\n" + user_input.strip()
    messages = [
        {
            "role": "user",
            "content": user_content
        }
    ]
    if assistant_output is not None:
        messages.append(
            {
                "role": "assistant",
                "content": assistant_output
            }
        )
    return messages
def process_func(example):
    # 完整聊天
    messages = build_messages(
        example["input"],
        example["output"]
    )
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )
    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    labels = input_ids.copy()
    # prompt（只有User）
    prompt_messages = build_messages(
        example["input"],
        None
    )
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=False
    )
    prompt_ids = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )["input_ids"]
    prompt_len = len(prompt_ids)
    # Mask User部分
    labels[:prompt_len] = [-100] * prompt_len
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [5]:
# ====================
# 3. 数据处理
# ====================
train_df = pd.read_json("./随机划分/训练文件/train_data_72_aug.jsonl", lines=True)
train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names)

Map:   0%|          | 0/4952 [00:00<?, ? examples/s]

In [6]:
# ====================
# 4. LoRA 配置
# ====================
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ],
    inference_mode=False,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 167,772,160 || all params: 7,415,795,712 || trainable%: 2.2624


In [7]:
# ====================
# 5. 训练参数
# ====================
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    
    num_train_epochs=7,
    
    learning_rate=8e-6,
    weight_decay=0.03,
    
    lr_scheduler_type="cosine",
    warmup_steps=0,
    
    logging_steps=50,
    save_steps=100,
    #eval_steps=100,
    #eval_strategy="steps",
    save_total_limit=2,
    
    #load_best_model_at_end=True,
    #metric_for_best_model="eval_loss",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    optim="adamw_torch"
)

In [8]:
# ====================
# 6. Trainer 执行
# ====================
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    #eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
50,0.486100
100,0.410800
150,0.381500
200,0.351100
250,0.337700
300,0.336400
350,0.317300
400,0.303900
450,0.300000
500,0.285300


TrainOutput(global_step=1085, training_loss=0.29070114408220565, metrics={'train_runtime': 2676.4437, 'train_samples_per_second': 12.952, 'train_steps_per_second': 0.405, 'total_flos': 2.897560657278075e+17, 'train_loss': 0.29070114408220565, 'epoch': 7.0})

In [9]:
def get_task_type(input_text):
    """
    根据input自动判断任务类型
    """
    text = input_text.lower()
    if "isobaric" in text:
        return "isobaric"
    elif "isothermal" in text:
        return "isothermal"
    else:
        raise ValueError("Cannot determine task type.")
def predict(example, model, tokenizer):
    model.eval()
    messages = build_messages(
        user_input=example["input"],
        assistant_output=None
    )
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()
    return response
def parse_vle_output(text, task_type):
    text = str(text)
    y1 = None
    T = None
    P = None
    m = re.search(
        r"Vapor-phase mole fraction of component 1\s*:\s*([-+]?\d*\.?\d+)",
        text,
        re.I
    )
    if m:
        y1 = float(m.group(1))
    if task_type == "isobaric":
        m = re.search(
            r"Equilibrium temperature\s*:\s*([-+]?\d*\.?\d+)",
            text,
            re.I
        )
        if m:
            T = float(m.group(1))
    else:
        m = re.search(
            r"Equilibrium pressure\s*:\s*([-+]?\d*\.?\d+)",
            text,
            re.I
        )
        if m:
            P = float(m.group(1))
    return y1, T, P
def batch_predict_mixed_vle(
    test_file_path,
    save_path,
    max_samples=None
):
    results = []
    # Overall
    true_y1_all = []
    pred_y1_all = []
    # Isobaric
    true_y1_isoB = []
    pred_y1_isoB = []
    true_T_isoB = []
    pred_T_isoB = []
    # Isothermal
    true_y1_isoT = []
    pred_y1_isoT = []
    true_P_isoT = []
    pred_P_isoT = []
    with open(test_file_path, "r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]
    if max_samples is not None:
        lines = lines[:max_samples]
    print(f"🚀 Total samples: {len(lines)}")
    model.eval()
    for i, line in enumerate(tqdm(lines)):
        try:
            data = json.loads(line)
            # 自动判断任务
            task_type = get_task_type(data["input"])
            # 模型预测
            raw_pred = predict(
                data,
                model,
                tokenizer
            )
            # 解析预测
            pred_y1, pred_T, pred_P = parse_vle_output(
                raw_pred,
                task_type
            )
            # 解析真实值
            true_y1, true_T, true_P = parse_vle_output(
                data["output"],
                task_type
            )
            #############################
            # 误差
            #############################
            err_y1 = None
            err_T = None
            err_P = None
            if pred_y1 is not None and true_y1 is not None:
                err_y1 = abs(pred_y1 - true_y1)
                true_y1_all.append(true_y1)
                pred_y1_all.append(pred_y1)
                if task_type == "isobaric":
                    true_y1_isoB.append(true_y1)
                    pred_y1_isoB.append(pred_y1)
                else:
                    true_y1_isoT.append(true_y1)
                    pred_y1_isoT.append(pred_y1)
            if task_type == "isobaric":
                if pred_T is not None and true_T is not None:
                    err_T = abs(pred_T - true_T)
                    true_T_isoB.append(true_T)
                    pred_T_isoB.append(pred_T)
            else:
                if pred_P is not None and true_P is not None:
                    err_P = abs(pred_P - true_P)
                    true_P_isoT.append(true_P)
                    pred_P_isoT.append(pred_P)

            #############################
            # 保存结果
            #############################
            result = {
                "task_type": task_type,
                "input": data["input"],
                "true_output": data["output"],
                "raw_prediction": raw_pred,
                "true_Y1": true_y1,
                "pred_Y1": pred_y1,
                "err_Y1": err_y1,
                "true_T": true_T,
                "pred_T": pred_T,
                "err_T": err_T,
                "true_P": true_P,
                "pred_P": pred_P,
                "err_P": err_P

            }
            results.append(result)
            #########################################
            print("\n" + "=" * 100)
            print(f"[{i+1}/{len(lines)}]")
            print("Task :", task_type)
            print("\nInput:")
            print(data["input"])
            print("\nGround Truth:")
            print(data["output"])
            print("\nPrediction:")
            print(raw_pred)
            print("\nParsed:")
            print(
                f"Y1 : {true_y1} -> {pred_y1}"
            )
            if task_type == "isobaric":
                print(
                    f"T  : {true_T} -> {pred_T}"
                )
            else:
                print(
                    f"P  : {true_P} -> {pred_P}"
                )
        except Exception as e:
            print(f"Sample {i+1} failed: {e}")
    #########################################
    # 保存json
    #########################################
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(
            results,
            f,
            indent=2,
            ensure_ascii=False
        )
    print(f"\nResults saved to {save_path}")
    #########################################
    # Metrics
    #########################################
    def print_metrics(name, y_true, y_pred):
        if len(y_true) == 0:
            print(f"\n{name}: No valid samples")
            return
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = r2_score(y_true, y_pred)
        print("\n" + "=" * 60)
        print(name)
        print(f"Samples : {len(y_true)}")
        print(f"MAE  : {mae:.6f}")
        print(f"RMSE : {rmse:.6f}")
        print(f"R²   : {r2:.6f}")
    print_metrics(
        "Overall Y1",
        true_y1_all,
        pred_y1_all
    )
    print_metrics(
        "Isobaric Y1",
        true_y1_isoB,
        pred_y1_isoB
    )
    print_metrics(
        "Isobaric Temperature",
        true_T_isoB,
        pred_T_isoB
    )
    print_metrics(
        "Isothermal Y1",
        true_y1_isoT,
        pred_y1_isoT
    )
    print_metrics(
        "Isothermal Pressure",
        true_P_isoT,
        pred_P_isoT
    )
    return results
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [10]:
test_file_path = "./随机划分/训练文件/test_data_72.jsonl"
save_path = "./随机划分/结果文件/test_data_72.json"
results_all = batch_predict_mixed_vle(
    test_file_path=test_file_path,
    save_path=save_path,
    max_samples= None
)

🚀 Total samples: 276


  0%|          | 1/276 [00:01<09:08,  1.99s/it]


[1/276]
Task : isobaric

Input:
Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: O
Component 2 SMILES: CC(=O)O
Operating pressure: 16.670 kPa
Liquid-phase mole fraction of component 1: 0.476
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature


Ground Truth:
Vapor-phase mole fraction of component 1: 0.586
Equilibrium temperature: 333.08 K

Prediction:
Vapor-phase mole fraction of component 1: 0.580
Equilibrium temperature: 334.99 K

Parsed:
Y1 : 0.586 -> 0.58
T  : 333.08 -> 334.99


  1%|          | 2/276 [00:03<08:33,  1.87s/it]


[2/276]
Task : isobaric

Input:
Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: CC(C)O
Component 2 SMILES: O
Operating pressure: 25.330 kPa
Liquid-phase mole fraction of component 1: 0.084
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature


Ground Truth:
Vapor-phase mole fraction of component 1: 0.484
Equilibrium temperature: 325.14 K

Prediction:
Vapor-phase mole fraction of component 1: 0.469
Equilibrium temperature: 324.99 K

Parsed:
Y1 : 0.484 -> 0.469
T  : 325.14 -> 324.99


...


[274/276]
Task : isobaric

Input:
Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: O
Component 2 SMILES: C(CN)N
Operating pressure: 13.330 kPa
Liquid-phase mole fraction of component 1: 0.834
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature


Ground Truth:
Vapor-phase mole fraction of component 1: 0.985
Equilibrium temperature: 332.19 K

Prediction:
Vapor-phase mole fraction of component 1: 0.986
Equilibrium temperature: 330.95 K

Parsed:
Y1 : 0.985 -> 0.986
T  : 332.19 -> 330.95


100%|█████████▉| 275/276 [08:11<00:01,  1.78s/it]


[275/276]
Task : isobaric

Input:
Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: CO
Component 2 SMILES: CCCC=O
Operating pressure: 98.660 kPa
Liquid-phase mole fraction of component 1: 0.654
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature


Ground Truth:
Vapor-phase mole fraction of component 1: 0.595
Equilibrium temperature: 332.55 K

Prediction:
Vapor-phase mole fraction of component 1: 0.602
Equilibrium temperature: 333.15 K

Parsed:
Y1 : 0.595 -> 0.602
T  : 332.55 -> 333.15


100%|██████████| 276/276 [08:13<00:00,  1.79s/it]


[276/276]
Task : isobaric

Input:
Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: CO
Component 2 SMILES: C1=CC=CC=C1
Operating pressure: 45.330 kPa
Liquid-phase mole fraction of component 1: 0.589
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature


Ground Truth:
Vapor-phase mole fraction of component 1: 0.582
Equilibrium temperature: 311.28 K

Prediction:
Vapor-phase mole fraction of component 1: 0.589
Equilibrium temperature: 311.59 K

Parsed:
Y1 : 0.582 -> 0.589
T  : 311.28 -> 311.59

Results saved to ./随机划分/结果文件/test_data_72.json

Overall Y1
Samples : 276
MAE  : 0.014928
RMSE : 0.026638
R²   : 0.992467

Isobaric Y1
Samples : 276
MAE  : 0.014928
RMSE : 0.026638
R²   : 0.992467

Isobaric Temperature
Samples : 276
MAE  : 1.768514
RMSE : 4.529008
R²   : 0.996392

Isothermal Y1: No valid samples

Isothermal Pressure: No valid samples


In [11]:
import pandas as pd
import re
import numpy as np
# =========================
# 1. 转成DataFrame
# =========================
df = pd.DataFrame(results_all)
# =========================
# 2. 从input提取信息
# =========================
def extract_info(input_text):
    input_text = str(input_text)
    smiles1 = None
    smiles2 = None
    x1 = None
    pressure = None
    temperature = None
    # Component 1
    m = re.search(
        r"Component 1 SMILES:\s*(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles1 = m.group(1).split("\n")[0].strip()
    # Component 2
    m = re.search(
        r"Component 2 SMILES:\s*(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles2 = m.group(1).split("\n")[0].strip()
    # Liquid composition
    m = re.search(
        r"Liquid-phase mole fraction of component 1:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        x1 = float(m.group(1))
    # Pressure
    m = re.search(
        r"Operating pressure:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        pressure = float(m.group(1))
    # Temperature
    m = re.search(
        r"Operating temperature:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        temperature = float(m.group(1))
    return pd.Series(
        [
            smiles1,
            smiles2,
            x1,
            pressure,
            temperature
        ],
        index=[
            "SMILES_1",
            "SMILES_2",
            "X1",
            "Input_P(kPa)",
            "Input_T(K)"
        ]
    )
# 提取信息
df[
    [
        "SMILES_1",
        "SMILES_2",
        "X1",
        "Input_P(kPa)",
        "Input_T(K)"
    ]
] = df["input"].apply(extract_info)
# =========================
# 3. 调整列顺序
# =========================
df_save = pd.DataFrame()
df_save["Task"] = df["task_type"]
df_save["SMILES_1"] = df["SMILES_1"]
df_save["SMILES_2"] = df["SMILES_2"]
df_save["Input_X1"] = df["X1"]
#df_save["Input_P(kPa)"] = df["Input_P(kPa)"]
#df_save["Input_T(K)"] = df["Input_T(K)"]
df_save["True_Y1"] = df["true_Y1"]
df_save["Pred_Y1"] = df["pred_Y1"]
df_save["Error_Y1"] = df["err_Y1"]
df_save["True_T(K)"] = df["true_T"]
df_save["Pred_T(K)"] = df["pred_T"]
df_save["Error_T(K)"] = df["err_T"]
#df_save["True_P(kPa)"] = df["true_P"]
#df_save["Pred_P(kPa)"] = df["pred_P"]
#df_save["Error_P(kPa)"] = df["err_P"]
df_save["Raw Prediction"] = df["raw_prediction"]
# ========================
# 4. 保存Excel
# =========================
save_path = "./随机划分/结果文件/excel结果文件/test_data_72.xlsx"
df_save.to_excel(
    save_path,
    index=False
)
print(f"保存成功：{save_path}")
df_save

保存成功：./随机划分/结果文件/excel结果文件/test_data_72.xlsx


,Task,SMILES_1,SMILES_2,Input_X1,True_Y1,Pred_Y1,Error_Y1,True_T(K),Pred_T(K),Error_T(K),Raw Prediction
0,isobaric,O,CC(=O)O,0.476,0.586,0.580,0.006,333.08,334.99,1.91,Vapor-phase mole fraction of component 1: 0.58...
1,isobaric,CC(C)O,O,0.084,0.484,0.469,0.015,325.14,324.99,0.15,Vapor-phase mole fraction of component 1: 0.46...
2,isobaric,CO,CCCC=O,0.713,0.751,0.650,0.101,336.75,334.15,2.60,Vapor-phase mole fraction of component 1: 0.65...
3,isobaric,CC(C=O)O,CO,0.849,0.788,0.774,0.014,327.65,328.55,0.90,Vapor-phase mole fraction of component 1: 0.77...
4,isobaric,O,C(C(CO)O)O,0.420,0.982,0.985,0.003,433.15,433.15,0.00,Vapor-phase mole fraction of component 1: 0.98...
...,...,...,...,...,...,...,...,...,...,...,...
271,isobaric,CC(C=O)O,CO,0.737,0.737,0.724,0.013,294.65,295.25,0.60,Vapor-phase mole fraction of component 1: 0.72...
272,isobaric,CC(=O)O,O,0.007,0.005,0.004,0.001,322.45,323.15,0.70,Vapor-phase mole fraction of component 1: 0.00...
273,isobaric,O,C(CN)N,0.834,0.985,0.986,0.001,332.19,330.95,1.24,Vapor-phase mole fraction of component 1: 0.98...
274,isobaric,CO,CCCC=O,0.654,0.595,0.602,0.007,332.55,333.15,0.60,Vapor-phase mole fraction of component 1: 0.60...


In [ ]:
import pandas as pd
import re
import numpy as np

# =========================
# 1. 转成DataFrame
# =========================
df = pd.DataFrame(results_all)

# =========================
# 2. 从input提取信息
# =========================
def extract_info(input_text):
    input_text = str(input_text)
    smiles1 = None
    smiles2 = None
    x1 = None
    pressure = None
    temperature = None
    m = re.search(
        r"Component 1 SMILES:(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles1 = m.group(1).split("\n")[0].strip()
    m = re.search(
        r"Component 2 SMILES:(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles2 = m.group(1).split("\n")[0].strip()
    m = re.search(
        r"Liquid-phase mole fraction of component 1:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        x1 = float(m.group(1))
    m = re.search(
        r"Operating pressure:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        pressure = float(m.group(1))
    m = re.search(
        r"Operating temperature:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        temperature = float(m.group(1))

    return pd.Series(
        [
            smiles1,
            smiles2,
            x1,
            pressure,
            temperature
        ]
    )

df[
    [
        "SMILES_1",
        "SMILES_2",
        "X1",
        "Input_P(kPa)",
        "Input_T(K)"
    ]
] = df["input"].apply(extract_info)
# =========================
# 3. 调整列顺序
# =========================
df_save = pd.DataFrame()
df_save["Task"] = df["task_type"]
df_save["SMILES_1"] = df["SMILES_1"]
df_save["SMILES_2"] = df["SMILES_2"]
df_save["Input_X1"] = df["X1"]
#df_save["Input_P(kPa)"] = df["Input_P(kPa)"]
#df_save["Input_T(K)"] = df["Input_T(K)"]
df_save["True_Y1"] = df["true_Y1"]
df_save["Pred_Y1"] = df["pred_Y1"]
df_save["Error_Y1"] = df["err_Y1"]
df_save["True_T(K)"] = df["true_T"]
df_save["Pred_T(K)"] = df["pred_T"]
df_save["Error_T(K)"] = df["err_T"]
#df_save["True_P(kPa)"] = df["true_P"]
#df_save["Pred_P(kPa)"] = df["pred_P"]
#df_save["Error_P(kPa)"] = df["err_P"]
#df_save["Raw Prediction"] = df["raw_prediction"]
# =========================
# 4. 保存Excel
# =========================
save_path = "./按体系划分/结果文件/excel结果文件/test_data_42_fine.xlsx"

df_save.to_excel(
    save_path,
    index=False
)
print(f"保存成功：{save_path}")
# 查看前几行
df_save

In [ ]:
test_file_path = "./随机划分/训练文件/train_data_42.jsonl"
save_path = "./随机划分/结果文件/train_result_72.json"
results_all = batch_predict_mixed_vle(
    test_file_path=test_file_path,
    save_path=save_path,
    max_samples= None
)

In [ ]:
import pandas as pd
import re
import numpy as np

# =========================
# 1. 转成DataFrame
# =========================
df = pd.DataFrame(results_all)

# =========================
# 2. 从input提取信息
# =========================
def extract_info(input_text):
    input_text = str(input_text)
    smiles1 = None
    smiles2 = None
    x1 = None
    pressure = None
    temperature = None
    m = re.search(
        r"Component 1 SMILES:(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles1 = m.group(1).split("\n")[0].strip()
    m = re.search(
        r"Component 2 SMILES:(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles2 = m.group(1).split("\n")[0].strip()
    m = re.search(
        r"Liquid-phase mole fraction of component 1:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        x1 = float(m.group(1))
    m = re.search(
        r"Operating pressure:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        pressure = float(m.group(1))
    m = re.search(
        r"Operating temperature:([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        temperature = float(m.group(1))

    return pd.Series(
        [
            smiles1,
            smiles2,
            x1,
            pressure,
            temperature
        ]
    )

df[
    [
        "SMILES_1",
        "SMILES_2",
        "X1",
        "Input_P(kPa)",
        "Input_T(K)"
    ]
] = df["input"].apply(extract_info)
# =========================
# 3. 调整列顺序
# =========================
df_save = pd.DataFrame()
df_save["Task"] = df["task_type"]
df_save["SMILES_1"] = df["SMILES_1"]
df_save["SMILES_2"] = df["SMILES_2"]
df_save["Input_X1"] = df["X1"]
df_save["Input_P(kPa)"] = df["Input_P(kPa)"]
df_save["Input_T(K)"] = df["Input_T(K)"]
df_save["True_Y1"] = df["true_Y1"]
df_save["Pred_Y1"] = df["pred_Y1"]
df_save["Error_Y1"] = df["err_Y1"]
df_save["True_T(K)"] = df["true_T"]
df_save["Pred_T(K)"] = df["pred_T"]
df_save["Error_T(K)"] = df["err_T"]
df_save["True_P(kPa)"] = df["true_P"]
df_save["Pred_P(kPa)"] = df["pred_P"]
df_save["Error_P(kPa)"] = df["err_P"]
df_save["Raw Prediction"] = df["raw_prediction"]
# =========================
# 4. 保存Excel
# =========================
save_path = "./随机划分/结果文件/excel结果文件/train_result_72.xlsx"

df_save.to_excel(
    save_path,
    index=False
)
print(f"保存成功：{save_path}")
# 查看前几行
df_save